In [1]:
import sys
import os
sys.path.insert(0, os.path.abspath('.'))

In [2]:
# Prepare dataset using some simulations
from Sim import Holdings
from Sim.runner import Runner
from Sim.metric_calculator import MetricCalculator
from Sim.trend_iter import TrendIter
from Sim.action_resolver import action_resolver 
from datetime import date
from Data.simulation_writer import writer as sim_writer
from Algorithm.MACDStrategy import MACDStrategy
from ML.ActionRanker.RwrdRealizedReturnActionRanker import RwrdRealizedReturnActionRanker

In [3]:
def sim( stk, days, subdir="", actionRanker=None ):
    for day in days:
        name = f"{stk}_{day}"
        funds = 10_000 
        d = date(month=4, day=int(day), year=2026)
        data_iter = TrendIter( stk, date=d ) 
        algorithm = MACDStrategy()
        holdings = Holdings( quantity=0, avg_price=0 )
        mc = MetricCalculator()
        mc.addMetrics(algorithm.getAlgoMetrics())
        runner = Runner(funds=funds, positions=holdings, algorithm=algorithm, metric_calculator=mc)
        runner.run(data_iter=data_iter, action_resolver=action_resolver )
        sim_writer(name, runner,subpath=subdir)
        if actionRanker:
            ranker = actionRanker(f"{name}", updated_name = f"{name}_ranked" )
            ranker.setReadBaseDir( f"simulation_results/{subdir}" )
            ranker.setWriteBaseDir( f"simulation_results/{subdir}/ranked" )
            ranker.rankAction()
        

In [4]:
days = [
    "6",
    "7",
    "8",
    "9",
    "10",
    "13",
    "14",
    "15",
    "16",
    "17",
]

stk_name = "ANET"
subdir = "DataDir"

In [5]:
#sim(stk_name, days, subdir, RwrdRealizedReturnActionRanker)

In [6]:
unranked = f"simulation_results/{subdir}"
ranked = f"simulation_results/{subdir}/ranked"

In [7]:
# We will inspect one of the data to see what ops we need to perform
# Below is Prediction kind of model
import pandas as pd

df = pd.read_csv(f"{unranked}/ANET_6.csv")

In [8]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 387 entries, 0 to 386
Data columns (total 13 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   action          387 non-null    object 
 1   close           387 non-null    float64
 2   open            387 non-null    float64
 3   high            387 non-null    float64
 4   low             387 non-null    float64
 5   volume          387 non-null    int64  
 6   price           387 non-null    float64
 7   time            387 non-null    object 
 8   macd_line       387 non-null    float64
 9   macd_signal     387 non-null    float64
 10  macd_histogram  387 non-null    float64
 11  rsi_14          387 non-null    float64
 12  vwap            387 non-null    float64
dtypes: float64(10), int64(1), object(2)
memory usage: 39.4+ KB


In [9]:
from Data.Operations import OperationBase
# We will use this custom operation to shift the closing price
class ShiftCol( OperationBase ):
    def __init__( self, cols= None, inplace = True ) -> None:
        super().__init__( cols )

    def operate( self, df:pd.DataFrame ) -> pd.DataFrame:
        df["future_price"] = df['close'].shift(-1)
        return df

In [10]:
# Test if it works
ShiftCol(cols = ["close"]).operate(df)

,action,close,open,high,low,volume,price,time,macd_line,macd_signal,macd_histogram,rsi_14,vwap,future_price
0,hold,127.739998,127.320000,127.970001,126.809998,93385,127.739998,2026-04-06 13:30:00+00:00,0.000000,0.000000,0.000000,50.000000,127.739998,127.904999
1,hold,127.904999,127.745003,128.000000,127.735001,4563,127.904999,2026-04-06 13:31:00+00:00,0.000000,0.000000,0.000000,50.000000,127.747685,128.000000
2,hold,128.000000,127.910004,128.000000,127.845001,1205,128.000000,2026-04-06 13:32:00+00:00,0.000000,0.000000,0.000000,50.000000,127.750751,127.919998
3,hold,127.919998,127.577301,127.919998,127.577301,1596,127.919998,2026-04-06 13:33:00+00:00,0.000000,0.000000,0.000000,50.000000,127.753432,127.834999
4,hold,127.834999,127.550003,127.834999,127.459999,8151,127.834999,2026-04-06 13:34:00+00:00,0.000000,0.000000,0.000000,50.000000,127.759537,127.940002
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
382,hold,126.055000,125.955002,126.150002,125.904999,28258,126.055000,2026-04-06 19:55:00+00:00,-0.014094,-0.016840,0.002746,56.452104,126.229422,126.019997
383,hold,126.019997,126.055000,126.084999,126.019997,22369,126.019997,2026-04-06 19:56:00+00:00,-0.009083,-0.015289,0.006206,53.052736,126.227379,126.110001
384,hold,126.110001,126.019997,126.120003,126.010002,28730,126.110001,2026-04-06 19:57:00+00:00,0.002126,-0.011806,0.013932,59.762164,126.225927,126.230003
385,hold,126.230003,126.120003,126.300003,126.120003,60130,126.230003,2026-04-06 19:58:00+00:00,0.020457,-0.005353,0.025810,66.613304,126.226030,126.269997


In [11]:
# We will add sma col 
from Data.Operations import AddSmaFiveMetricColumn
AddSmaFiveMetricColumn().operate(df)

,sma
0,0.000000
1,0.000000
2,0.000000
3,0.000000
4,127.879999
...,...
382,125.966002
383,125.988000
384,126.022000
385,126.074001


In [12]:
# We will remove some cols 
cols = [
    "open",
    "high",
    "low",
    "volume",
    "close",
    "time",
    "action"
]
from Data.Operations import RemoveColumns

RemoveColumns(cols=cols).operate(df)

,macd_line,future_price,vwap,macd_histogram,macd_signal,rsi_14,price
0,0.000000,127.904999,127.739998,0.000000,0.000000,50.000000,127.739998
1,0.000000,128.000000,127.747685,0.000000,0.000000,50.000000,127.904999
2,0.000000,127.919998,127.750751,0.000000,0.000000,50.000000,128.000000
3,0.000000,127.834999,127.753432,0.000000,0.000000,50.000000,127.919998
4,0.000000,127.940002,127.759537,0.000000,0.000000,50.000000,127.834999
...,...,...,...,...,...,...,...
382,-0.014094,126.019997,126.229422,0.002746,-0.016840,56.452104,126.055000
383,-0.009083,126.110001,126.227379,0.006206,-0.015289,53.052736,126.019997
384,0.002126,126.230003,126.225927,0.013932,-0.011806,59.762164,126.110001
385,0.020457,126.269997,126.226030,0.025810,-0.005353,66.613304,126.230003


In [13]:
# Finally we will clean up
from Data.Operations import RemoveEmptyNullRows
tmp = RemoveEmptyNullRows().operate(df)

In [14]:
tmp.isnull().sum()

action            0
close             0
open              0
high              0
low               0
volume            0
price             0
time              0
macd_line         0
macd_signal       0
macd_histogram    0
rsi_14            0
vwap              0
future_price      0
dtype: int64

In [15]:
# So the sequence of operations looks like this
ops = [
    ShiftCol(cols = ["close"]),
    AddSmaFiveMetricColumn(),
    RemoveColumns(cols=cols),
    RemoveEmptyNullRows(),
]

In [16]:
# We will combine all the data and format the data,
# By format i mean we will also apply column operations

from Data.data_formatter import collect_and_format_data

final = collect_and_format_data(
    "DataDir",
    ops,
    write_csv=True,
    name = "Combined_ANET_April",
)

C:\Users\Meowmaster\AppData\Local\Temp\ipykernel_23896\3513733785.py:8: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df["future_price"] = df['close'].shift(-1)
c:\Users\Meowmaster\Desktop\PROject\IntradaySimulator\Data\data_formatter.py:42: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df[ operatedColumn.columns ] = operatedColumn
C:\Users\Meowmaster\AppData\Local\Temp\ipykernel_23896\3513733785.py:8: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc

In [17]:
# This is the final dataset
final.info()

<class 'pandas.core.frame.DataFrame'>
Index: 3886 entries, 0 to 3894
Data columns (total 8 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   macd_line       3886 non-null   float64
 1   future_price    3886 non-null   float64
 2   sma             3886 non-null   float64
 3   vwap            3886 non-null   float64
 4   macd_histogram  3886 non-null   float64
 5   macd_signal     3886 non-null   float64
 6   rsi_14          3886 non-null   float64
 7   price           3886 non-null   float64
dtypes: float64(8)
memory usage: 273.2 KB


In [18]:
# To train the model itself we need a scaler
from ML.Scaler import STDScaler
from sklearn.model_selection import train_test_split

scaler = STDScaler()

In [19]:
# We will use future_price as our prediction column
X = final.drop(columns=['future_price'])
y = final['future_price']

In [20]:
X.info()

<class 'pandas.core.frame.DataFrame'>
Index: 3886 entries, 0 to 3894
Data columns (total 7 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   macd_line       3886 non-null   float64
 1   sma             3886 non-null   float64
 2   vwap            3886 non-null   float64
 3   macd_histogram  3886 non-null   float64
 4   macd_signal     3886 non-null   float64
 5   rsi_14          3886 non-null   float64
 6   price           3886 non-null   float64
dtypes: float64(7)
memory usage: 242.9 KB


In [21]:
from ML.Trainer import SVRPredictor

trainer = SVRPredictor( X, y )
trainer.setScaler(scaler)

In [22]:
trainer.train()

In [23]:
trainer.test()

{'mse': 0.29482500371291676, 'rmse': np.float64(0.5429779035217885)}

In [24]:
# If your cringing at using training data for prediction then I am sorry mate
selected = X.iloc[[10, 20]]
new_data = selected.to_dict('records')

In [25]:
trainer.predict(new_data)

array([146.60864701, 147.35508844])

In [26]:
# Save the model for future use
trainer.save("svr")

In [27]:
# Will Test we can retrive the same stuff back
new_trainer  =  SVRPredictor.load_only()

In [28]:
new_trainer.load("svr")

In [29]:
new_trainer.predict(new_data)

array([146.60864701, 147.35508844])

In [35]:
# Below will be decider model, similar stuff

cols = [
    "open",
    "high",
    "low",
    "volume",
    "close",
    "time",
    "action"
]

ops = [
    AddSmaFiveMetricColumn(),
    RemoveColumns(cols=cols),
    RemoveEmptyNullRows(),
]

df = collect_and_format_data(
    "DataDir/ranked",
    ops,
    write_csv=True,
    name = "Combined_ANET_April_ranked",
)
X = df.drop(columns=['action_quality'])
y = df['action_quality']

c:\Users\Meowmaster\Desktop\PROject\IntradaySimulator\Data\data_formatter.py:42: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df[ operatedColumn.columns ] = operatedColumn
c:\Users\Meowmaster\Desktop\PROject\IntradaySimulator\Data\data_formatter.py:42: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df[ operatedColumn.columns ] = operatedColumn
c:\Users\Meowmaster\Desktop\PROject\IntradaySimulator\Data\data_formatter.py:42: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a Data

In [36]:
from ML.Scaler import WeightedLabelScaler

scaler = WeightedLabelScaler()

In [37]:
from ML.Trainer import RandomForestDecider

trainer = RandomForestDecider( X, y )
trainer.setScaler(scaler)

In [38]:
trainer.train()


Columns in training data before scaling: ['macd_line', 'vwap', 'sma', 'macd_histogram', 'macd_signal', 'rsi_14', 'price']


In [39]:
trainer.test()

{'accuracy': 0.87782340862423}

In [40]:
selected = X.iloc[[10, 20]]
new_data = selected.to_dict('records')
trainer.predict(new_data)

array(['good', 'good'], dtype=object)

In [41]:
trainer.save("rfd")

In [42]:
new_trainer = RandomForestDecider.load_only()
new_trainer.load('rfd')
new_trainer.predict(new_data)

array(['good', 'good'], dtype=object)